<a href="https://colab.research.google.com/github/Aris-1712/LLM-fine-tuning/blob/main/PreferrenceBasedTraining_DPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install dependencies

!pip install transformers
!pip install datasets
!pip install accelerate
!pip install peft
!pip install trl
!pip install bitsandbytes
!pip install pyMuPdf

In [ ]:
# load tokenzier to convert strint to tokens

from transformers import AutoTokenizer
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# tokenizer should be of the model itself
tokenizer = AutoTokenizer.from_pretrained(model)

# set pad token and eos (end of sentence) token
if tokenizer.pad_token == None:
  tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# dataset
from datasets import load_dataset

data = load_dataset("json", data_files="/content/DPO_training_dataset.json", split= "train")


In [ ]:
data[0]

In [ ]:
# Load the Base Model

from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model, device_map="auto")

In [ ]:
from peft import LoraConfig, TaskType
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)

In [ ]:
# Import exisisting Non Instructional Trained Model

import zipfile

# Open the ZIP file in read mode ('r')
with zipfile.ZipFile('instructional_trained_model.zip', 'r') as zip_ref:
    # Extract all contents to the current directory
    zip_ref.extractall('tinyllama-lora')

In [ ]:
# crete the peft model using the lora config

from peft import get_peft_model, PeftModel

path = "/content/tinyllama-lora/instructionalTraining/checkpoint-16"

# Load the base model and non instructional model by merging the weights
InstructionTrainingModel = PeftModel.from_pretrained(model, path)
InstructionTrainingModel = InstructionTrainingModel.merge_and_unload()

peft_model = get_peft_model(InstructionTrainingModel, lora_config)

In [ ]:
# DPO training configs

from trl import DPOTrainer, DPOConfig


dpo_args = DPOConfig(
    output_dir="./tinyllama-preference-alignment",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    beta=0.01,
    report_to=[],
    logging_dir=None, # disable logging to wandb or tensorboard
    loss_type="sigmoid",  # or "hinge", depending on experiment
    remove_unused_columns=False
)


trainer = DPOTrainer(
    model=peft_model,
    ref_model=None,
    args=dpo_args,
    train_dataset=data,
    processing_class=tokenizer,   # instead of tokenizer argument
    # you can pass data_collator if needed,
    # optionally eval_dataset etc.
)


trainer.train()


In [ ]:
from transformers import AutoModelForCausalLM

pathPreference = "/content/tinyllama-preference-alignment/checkpoint-25"

modelPreference = AutoModelForCausalLM.from_pretrained(pathPreference, device_map="auto")


In [ ]:
prompt_text = "What does the change in Alphabet\u2019s net income indicate about its operational performance? (case 2)"
prompt = f"<s>[INST] {prompt_text} [/INST]"

inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

outputs = modelPreference.generate(
    **inputs,
    max_new_tokens=200, # Increased max_new_tokens
    temperature=0.1,    # Decreased temperature for less randomness
    top_p=0.9,
    do_sample=True,
    repetition_penalty=2.0 # Increased repetition penalty
)


decoded_output = tokenizer.decode(outputs[0],skip_special_tokens=True) # Removed skip_special_tokens=True
print(f"\nDecoded Output (including special tokens):\n{decoded_output}")